In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd



In [2]:
df = gpd.read_file('./data/prostor/blocks_investment.geojson')
df

,land_use,land_area,built_area,land_cost,construction_cost,investment_need,NPV,IRR,ROI,PP_years,EI,spatial_potential,INV,geometry
0,AGRICULTURE,202399.73,12550.42,8.195883e+07,3.137604e+08,3.957192e+08,-2.266165e+08,0.00,0.92,NaN,0.00,3.0,21.36,"POLYGON ((388511.373 6644951.946, 388558.037 6..."
1,BUSINESS,201529.49,40777.55,1.457553e+08,2.242765e+09,2.388521e+09,2.259647e+08,0.14,4.81,12.87,45.55,2.0,26.09,"POLYGON ((388558.037 6645102.374, 388511.373 6..."
2,TRANSPORT,164663.86,24985.66,7.927197e+07,4.497418e+08,5.290138e+08,-5.605498e+07,0.10,3.83,NaN,0.00,4.0,42.73,"POLYGON ((387957.434 6643444.937, 388007.492 6..."
3,RESIDENTIAL,161571.83,72568.71,2.486372e+08,3.265592e+09,3.514229e+09,1.204906e+09,0.27,2.51,4.11,91.47,4.0,95.11,"POLYGON ((388007.492 6643608.409, 387957.434 6..."
4,RESIDENTIAL,49.93,2.17,7.532216e+04,9.764970e+04,1.729719e+05,-3.185744e+04,0.04,1.14,NaN,0.00,4.0,42.73,"POLYGON ((388000.75 6643199.606, 388001.306 66..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,BUSINESS,606.98,203.64,5.072911e+05,1.120007e+07,1.170736e+07,2.220737e+06,0.16,5.56,11.41,10.36,2.0,5.94,"POLYGON ((390950.379 6643779.96, 390941.389 66..."
63,AGRICULTURE,2.67,NaN,1.102000e+01,NaN,1.102000e+01,-2.750999e+04,0.03,1.64,NaN,0.00,3.0,21.36,"POLYGON ((391069.678 6644283.933, 391072.529 6..."
64,TRANSPORT,4307.66,688.58,2.103883e+06,1.239436e+07,1.449824e+07,-1.169555e+06,0.11,4.01,NaN,0.00,4.0,42.73,"POLYGON ((388222.552 6643218.262, 388215.418 6..."
65,BUSINESS,61.63,13.49,5.911855e+04,7.418000e+05,8.009186e+05,1.215635e+05,0.15,5.09,11.93,7.90,2.0,4.52,"POLYGON ((388314.421 6645193.77, 388311.871 66..."


## Версия 1

In [3]:
import pandas as pd
import numpy as np

# ------- РАСЧЁТ -------
def compute_ser(df: pd.DataFrame,
                params: dict,
                land_df: pd.DataFrame | None = None) -> dict:
    """
    Возвращает словарь с ключами основных и доп. показателей СЭР.
    """

    P        = params['population']
    Emp_base = params['employment_base']
    wage_base= params['avg_wage_base']
    FA_base  = params.get('FA_base', 0.0)
    Dep_base = params.get('Dep_base', 0.0)
    T_build  = params.get('build_years', 1)

    tax      = params['tax_rates']                  # {'pit','cit','prop','land'}
    k_VA_b   = params['va_coeff_build']
    y_m2     = params['va_per_m2_ops']
    dens     = params['jobs_per_m2']
    wage_by  = params['wage_by_use']
    prof_sh  = params.get('profit_share_ops', {})
    cap_share= params.get('capex_capitalizable_share', {})
    a_use    = params.get('amortization_rates', {})
    build_wage_share   = params.get('build_wage_share', 0.25)
    build_profit_margin= params.get('build_profit_margin', 0.05)

    d = df.copy()
    for col in ['land_area','built_area','land_cost','construction_cost',
                'investment_need','NPV','ROI','EI','INV']:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors='coerce')

    # Инвестиции
    I_total = d['investment_need'].sum()
    inv_capex_pc_build_total = I_total / max(P, 1)

    # Группировка
    g = d.groupby('land_use', dropna=False).agg(
        I=('investment_need','sum'),
        built=('built_area','sum')
    ).reset_index()

    def get_by_use(u, mapping, default=0.0):
        return mapping.get(u, mapping.get('default', default))

    # ВРП: стройка
    g['k_va_build'] = g['land_use'].apply(lambda u: get_by_use(u, k_VA_b, 0.0))
    VA_build_annual = (g['I'] * g['k_va_build']).sum() / max(T_build,1)
    grp_pc_build_annual = VA_build_annual / max(P,1)

    # ВРП: эксплуатация
    g['y_m2'] = g['land_use'].apply(lambda u: get_by_use(u, y_m2, 0.0))
    VA_ops_annual = (g['built'] * g['y_m2']).sum()
    grp_pc_ops_annual = VA_ops_annual / max(P,1)

    # Бюджет: стройка
    W_build_annual = (I_total / max(T_build,1)) * build_wage_share
    PIT_build_annual = W_build_annual * 12 * tax['pit']
    CIT_build_annual = (I_total / max(T_build,1)) * build_profit_margin * tax['cit']
    budget_revenue_build_annual = PIT_build_annual + CIT_build_annual

    # Бюджет: эксплуатация
    g['jobs_density'] = g['land_use'].apply(lambda u: get_by_use(u, dens, 0.0))
    g['jobs'] = g['built'] * g['jobs_density']
    g['wage_month'] = g['land_use'].apply(lambda u: get_by_use(u, wage_by, 0.0))
    PIT_ops_annual = (g['jobs'] * g['wage_month'] * 12).sum() * tax['pit']

    g['va_ops'] = g['built'] * g['y_m2']
    g['profit_share'] = g['land_use'].apply(lambda u: get_by_use(u, prof_sh, 0.0))
    CIT_ops_annual = (g['va_ops'] * g['profit_share']).sum() * tax['cit']

    g['cap_share'] = g['land_use'].apply(lambda u: get_by_use(u, cap_share, 1.0))
    FA_add = (g['I'] * g['cap_share']).sum()
    Property_tax_annual = FA_add * tax['prop']

    land_tax_delta_annual = 0.0
    if land_df is not None and not land_df.empty:
        ld = land_df.copy()
        for c in ['cad_before','cad_after']:
            ld[c] = pd.to_numeric(ld[c], errors='coerce').fillna(0.0)
        land_tax_delta_annual = (ld['cad_after'].sum() - ld['cad_before'].sum()) * tax['land']

    budget_revenue_ops_annual = PIT_ops_annual + CIT_ops_annual + Property_tax_annual + land_tax_delta_annual

    # Средняя зарплата
    Jobs_new = g['jobs'].sum()
    if Emp_base + Jobs_new > 0:
        avg_wage_new = (wage_base * Emp_base + (g['jobs'] * g['wage_month']).sum()) / (Emp_base + Jobs_new)
    else:
        avg_wage_new = wage_base
    avg_wage_delta = avg_wage_new - wage_base

    # Амортизация и износ
    g['a_use'] = g['land_use'].apply(lambda u: get_by_use(u, a_use, 0.03))
    wear_amount_annual = (g['I'] * g['cap_share'] * g['a_use']).sum()
    FA_new  = FA_base + FA_add
    Dep_new = Dep_base + wear_amount_annual
    wear_percent_new = Dep_new / max(FA_new - wear_amount_annual, 1.0)

    return {
        # ОСНОВНЫЕ
        'main': {
            'inv_capex_pc_build_total': float(inv_capex_pc_build_total),  # ₽/чел за период стройки
            'grp_pc_build_annual':      float(grp_pc_build_annual),       # ₽/чел/год в стройке
            'grp_pc_ops_annual':        float(grp_pc_ops_annual),         # ₽/чел/год в эксплуатации
            'budget_revenue_build_annual': float(budget_revenue_build_annual),  # ₽/год
            'budget_revenue_ops_annual':   float(budget_revenue_ops_annual),    # ₽/год
            'avg_wage_new':             float(avg_wage_new),              # ₽/мес
            'avg_wage_delta':           float(avg_wage_delta),            # ₽/мес
        },
        # ДОПОЛНИТЕЛЬНЫЕ
        'extra': {
            'jobs_created':             float(Jobs_new),                  # чел
            'FA_added':                 float(FA_add),                    # ₽
            'property_tax_annual':      float(Property_tax_annual),       # ₽/год
            'land_tax_delta_annual':    float(land_tax_delta_annual),     # ₽/год
            'wear_amount_annual':       float(wear_amount_annual),        # ₽/год
            'wear_percent_new':         float(wear_percent_new)           # доля
        }
    }

# ------- ФОРМАТИРОВАННЫЙ ВЫВОД -------
def format_ser_output(result: dict) -> str:
    """Возвращает читаемую строку с разделением на основные и дополнительные."""
    main = result['main']; extra = result['extra']

    def fmt_rub(x):   return f"{x:,.0f} ₽".replace(',', ' ')
    def fmt_rub1(x):  return f"{x:,.1f} ₽".replace(',', ' ')
    def fmt_pc(x):    return f"{x*100:.1f}%"
    def fmt_num(x):   return f"{x:,.0f}".replace(',', ' ')

    lines = []
    lines.append("ОСНОВНЫЕ ПОКАЗАТЕЛИ:")
    lines.append(f"  Инвестиции на человека (за стройку): {fmt_rub1(main['inv_capex_pc_build_total'])}/чел")
    lines.append(f"  ВРП на душу в год стройки:           {fmt_rub1(main['grp_pc_build_annual'])}/чел/год")
    lines.append(f"  ВРП на душу в эксплуатации:          {fmt_rub1(main['grp_pc_ops_annual'])}/чел/год")
    lines.append(f"  Доходы бюджета в стройку:            {fmt_rub(main['budget_revenue_build_annual'])}/год")
    lines.append(f"  Доходы бюджета в эксплуатацию:       {fmt_rub(main['budget_revenue_ops_annual'])}/год")
    lines.append(f"  Новая средняя зарплата:              {fmt_rub1(main['avg_wage_new'])}/мес")
    lines.append(f"  Прирост средней зарплаты:            {fmt_rub1(main['avg_wage_delta'])}/мес")

    lines.append("\nДОПОЛНИТЕЛЬНЫЕ ПОКАЗАТЕЛИ:")
    lines.append(f"  Создано рабочих мест:                {fmt_num(extra['jobs_created'])} чел")
    lines.append(f"  Прирост основных средств:            {fmt_rub(extra['FA_added'])}")
    lines.append(f"  Налог на имущество (год):            {fmt_rub(extra['property_tax_annual'])}")
    lines.append(f"  Прирост земельного налога (год):     {fmt_rub(extra['land_tax_delta_annual'])}")
    lines.append(f"  Амортизация новых ОС (год):          {fmt_rub(extra['wear_amount_annual'])}")
    lines.append(f"  Новый процент износа:                {fmt_pc(extra['wear_percent_new'])}")

    return "\n".join(lines)



In [4]:

example_params = {
    'population': 500_000,
    'employment_base': 230_000,
    'avg_wage_base': 70_000,  # ₽/мес
    'FA_base': 300_000_000_000,
    'Dep_base': 120_000_000_000,
    'build_years': 3,
    'tax_rates': {'pit': 0.13, 'cit': 0.17, 'prop': 0.02, 'land': 0.013},

    # доля VA от инвестиций в год строительства
    'va_coeff_build': {
        'BUSINESS': 0.50, 'RESIDENTIAL': 0.45, 'TRANSPORT': 0.55, 'AGRICULTURE': 0.45, 'SPECIAL': 0.50, 'default': 0.50
    },
    # VA/м²·год в эксплуатации
    'va_per_m2_ops': {
        'BUSINESS': 12000.0, 'RESIDENTIAL': 2000.0, 'TRANSPORT': 5000.0, 'AGRICULTURE': 3000.0, 'SPECIAL': 7000.0, 'default': 0.0
    },
    # Плотность рабочих мест, чел/м²
    'jobs_per_m2': {
        'BUSINESS': 1/18, 'RESIDENTIAL': 0.0, 'TRANSPORT': 1/90, 'AGRICULTURE': 1/400, 'SPECIAL': 1/30, 'default': 0.0
    },
    # Зарплаты по видам, ₽/мес
    'wage_by_use': {
        'BUSINESS': 85_000, 'RESIDENTIAL': 0, 'TRANSPORT': 60_000, 'AGRICULTURE': 45_000, 'SPECIAL': 75_000, 'default': 0
    },
    # Доля прибыли в VA
    'profit_share_ops': {
        'BUSINESS': 0.18, 'TRANSPORT': 0.10, 'AGRICULTURE': 0.08, 'SPECIAL': 0.15, 'RESIDENTIAL': 0.00, 'default': 0.12
    },
    # Доля капвложений, капитализируемая в ОС
    'capex_capitalizable_share': {
        'BUSINESS': 0.95, 'RESIDENTIAL': 0.95, 'TRANSPORT': 0.90, 'AGRICULTURE': 0.90, 'SPECIAL': 0.95, 'default': 0.95
    },
    # Нормы амортизации по видам
    'amortization_rates': {
        'BUSINESS': 0.03, 'RESIDENTIAL': 0.03, 'TRANSPORT': 0.06, 'AGRICULTURE': 0.05, 'SPECIAL': 0.04, 'default': 0.03
    },
    # Стройка
    'build_wage_share': 0.25,
    'build_profit_margin': 0.05
}
# ------- ПРИМЕР ВЫЗОВА -------
result = compute_ser(df, example_params, None)
result

{'main': {'inv_capex_pc_build_total': 116188.72780864002,
  'grp_pc_build_annual': 18535.417544199,
  'grp_pc_ops_annual': 12788.7832,
  'budget_revenue_build_annual': 7716868005.290508,
  'budget_revenue_ops_annual': 4162861090.646717,
  'avg_wage_new': 70807.29555860779,
  'avg_wage_delta': 807.2955586077878},
 'extra': {'jobs_created': 23801.638791666664,
  'FA_added': 54942151244.5025,
  'property_tax_annual': 1098843024.8900502,
  'land_tax_delta_annual': 0.0,
  'wear_amount_annual': 1878673263.03304,
  'wear_percent_new': 0.3452032874083618}}

In [5]:
print(format_ser_output(result))

ОСНОВНЫЕ ПОКАЗАТЕЛИ:
  Инвестиции на человека (за стройку): 116 188.7 ₽/чел
  ВРП на душу в год стройки:           18 535.4 ₽/чел/год
  ВРП на душу в эксплуатации:          12 788.8 ₽/чел/год
  Доходы бюджета в стройку:            7 716 868 005 ₽/год
  Доходы бюджета в эксплуатацию:       4 162 861 091 ₽/год
  Новая средняя зарплата:              70 807.3 ₽/мес
  Прирост средней зарплаты:            807.3 ₽/мес

ДОПОЛНИТЕЛЬНЫЕ ПОКАЗАТЕЛИ:
  Создано рабочих мест:                23 802 чел
  Прирост основных средств:            54 942 151 245 ₽
  Налог на имущество (год):            1 098 843 025 ₽
  Прирост земельного налога (год):     0 ₽
  Амортизация новых ОС (год):          1 878 673 263 ₽
  Новый процент износа:                34.5%


## Версия 2

In [7]:
import pandas as pd
import numpy as np

def ser_deltas_min(df: pd.DataFrame, params: dict, pretty: bool = True) -> pd.DataFrame:
    # ====== базовые ======
    P        = params['population']
    Emp_base = params['employment_base']
    W_base   = params['avg_wage_base']

    T_build  = params.get('build_years', 3)
    tax = params.get('tax_rates', {'pit': 0.13, 'cit': 0.17, 'prop': 0.02})
    pit, cit, prop = tax.get('pit', 0.13), tax.get('cit', 0.17), tax.get('prop', 0.02)

    va_build = params.get('va_coeff_build', {
        'BUSINESS': 0.50, 'RESIDENTIAL': 0.45, 'TRANSPORT': 0.55,
        'AGRICULTURE': 0.45, 'SPECIAL': 0.50, 'default': 0.50
    })
    va_m2 = params.get('va_per_m2_ops', {
        'BUSINESS': 12000.0, 'RESIDENTIAL': 2000.0, 'TRANSPORT': 5000.0,
        'AGRICULTURE': 3000.0, 'SPECIAL': 7000.0, 'default': 0.0
    })
    jobs_m2 = params.get('jobs_per_m2', {
        'BUSINESS': 1/18, 'RESIDENTIAL': 0.0, 'TRANSPORT': 1/90,
        'AGRICULTURE': 1/400, 'SPECIAL': 1/30, 'default': 0.0
    })
    wage_by = params.get('wage_by_use', {
        'BUSINESS': 85_000, 'RESIDENTIAL': 0, 'TRANSPORT': 60_000,
        'AGRICULTURE': 45_000, 'SPECIAL': 75_000, 'default': 0
    })
    profit_sh = params.get('profit_share_ops', {
        'BUSINESS': 0.18, 'TRANSPORT': 0.10, 'AGRICULTURE': 0.08,
        'SPECIAL': 0.15, 'RESIDENTIAL': 0.00, 'default': 0.12
    })
    cap_share = params.get('capex_capitalizable_share', {
        'BUSINESS': 0.95, 'RESIDENTIAL': 0.95, 'TRANSPORT': 0.90,
        'AGRICULTURE': 0.90, 'SPECIAL': 0.95, 'default': 0.95
    })
    amort = params.get('amortization_rates', {
        'BUSINESS': 0.03, 'RESIDENTIAL': 0.03, 'TRANSPORT': 0.06,
        'AGRICULTURE': 0.05, 'SPECIAL': 0.04, 'default': 0.03
    })
    build_wage_share    = params.get('build_wage_share', 0.25)
    build_profit_margin = params.get('build_profit_margin', 0.05)

    # ====== подготовка ======
    d = df.copy()
    for col in ['built_area','investment_need']:
        d[col] = pd.to_numeric(d[col], errors='coerce').fillna(0.0)
    d['land_use'] = d['land_use'].astype(str)

    g = d.groupby('land_use', dropna=False).agg(I=('investment_need','sum'),
                                                A=('built_area','sum')).reset_index()
    get = lambda mp, k, default=0.0: mp.get(k, mp.get('default', default))

    # ====== расчёты (дельты) ======
    I_total = g['I'].sum()
    delta_invcap_pc_build = I_total / max(P, 1)

    g['k_va_build'] = g['land_use'].apply(lambda u: get(va_build, u, 0.0))
    VA_build_annual = (g['I'] * g['k_va_build']).sum() / max(T_build, 1)
    delta_grp_pc_build = VA_build_annual / max(P, 1)

    g['y_m2'] = g['land_use'].apply(lambda u: get(va_m2, u, 0.0))
    VA_ops_annual = (g['A'] * g['y_m2']).sum()
    delta_grp_pc_ops = VA_ops_annual / max(P, 1)

    W_build_annual   = (I_total / max(T_build,1)) * build_wage_share
    PIT_build_annual = W_build_annual * 12 * pit
    CIT_build_annual = (I_total / max(T_build,1)) * build_profit_margin * cit
    delta_budget_build = PIT_build_annual + CIT_build_annual

    g['jobs_m2'] = g['land_use'].apply(lambda u: get(jobs_m2, u, 0.0))
    g['jobs']    = g['A'] * g['jobs_m2']
    g['wage']    = g['land_use'].apply(lambda u: get(wage_by, u, 0.0))
    PIT_ops_annual = (g['jobs'] * g['wage'] * 12).sum() * pit

    g['profit_sh'] = g['land_use'].apply(lambda u: get(profit_sh, u, 0.0))
    CIT_ops_annual = (g['A'] * g['y_m2'] * g['profit_sh']).sum() * cit

    g['cap_share'] = g['land_use'].apply(lambda u: get(cap_share, u, 1.0))
    FA_add = (g['I'] * g['cap_share']).sum()
    Property_tax_annual = FA_add * prop
    delta_budget_ops = PIT_ops_annual + CIT_ops_annual + Property_tax_annual

    Jobs_new = g['jobs'].sum()
    if Emp_base + Jobs_new > 0:
        W_new = (W_base * Emp_base + (g['jobs'] * g['wage']).sum()) / (Emp_base + Jobs_new)
        delta_wage = W_new - W_base
    else:
        delta_wage = 0.0

    g['a'] = g['land_use'].apply(lambda u: get(amort, u, 0.03))
    Dep_add_annual = (g['I'] * g['cap_share'] * g['a']).sum()
    delta_wear_thousand_rub = Dep_add_annual / 1_000.0  # тыс. руб.

    out = pd.DataFrame([
        ["Объём инвестиций в основной капитал на душу населения",
         delta_invcap_pc_build, np.nan],
        ["Валовый региональный продукт на душу населения",
         delta_grp_pc_build,   delta_grp_pc_ops],
        ["Доходы бюджета территории",
         delta_budget_build,   delta_budget_ops],
        ["Средний уровень заработной платы",
         np.nan,               delta_wage],
        ["Износ основного фонда (тыс. руб.)",
         np.nan,               delta_wear_thousand_rub],
    ], columns=['indicator','delta_build_year','delta_ops_year'])

    if not pretty:
        return out

    # ====== форматирование чисел: без экспоненты, с пробелами
    def fmt(x):
        if pd.isna(x): return ""
        v = float(x)
        if abs(v) >= 100:   # крупные числа без десятых
            return f"{v:,.0f}".replace(",", " ")
        elif abs(v) >= 1:
            return f"{v:,.2f}".replace(",", " ")
        else:
            return f"{v:,.4f}".replace(",", " ")

    for col in ['delta_build_year','delta_ops_year']:
        out[col] = out[col].map(fmt)

    return out

# Пример:
# result_df = ser_deltas_min(df, {
#     'population': 500_000,
#     'employment_base': 230_000,
#     'avg_wage_base': 70_000
# })
# print(result_df)




result_df = ser_deltas_min(df, {
    'population': 500_000,
    'employment_base': 230_000,
    'avg_wage_base': 70_000
    # остальные параметры можно не передавать — будут дефолты
})
result_df

,indicator,delta_build_year,delta_ops_year
0,Объём инвестиций в основной капитал на душу на...,116 189,
1,Валовый региональный продукт на душу населения,18 535,12 789
2,Доходы бюджета территории,7 716 868 005,4 162 861 091
3,Средний уровень заработной платы,,807
4,Износ основного фонда (тыс. руб.),,1 878 673
